# Train YOLO Detect on Colab (T4) - `detectset_split.zip` (SELECTED 6-class, STANDARD overwrite baseline)

**Classes (`SELECTED`, IDs = order):** `cell phone` (0), `laptop` (1), `wallet` (2), `bottle` (3), `bag` (4, merges old `handbag` + `backpack`), `cup` (5). Edit `SELECTED` in the setup cell per run - everything downstream derives from it. Keep identical to the grafted notebook for a fair ablation.

**Base:** `yolo11x.pt` (COCO). STANDARD baseline for the graft ablation: trains straight from the base, letting Ultralytics do its default `nc=80 -> nc=6` head reset (fresh random classifier, backbone/neck transferred). Identical data, schedule, and hyperparameters to the grafted notebook - the head init is the only variable. Compare per-class AP50/AP50-95/recall at ~10 epochs and at convergence.

**Dataset:** local `src/datasets/detectset_split/` zipped to `Drive/Thesis_Workspace/datasets/detectset_split.zip` (`images/train 3070 = 1475 vid* + 1595 Angle1/2`, `val 170`, `test 171`; `data.yaml` names in `SELECTED` order). The stage cell unzips to `/content/datasets/detectset_split/`, validates pairing + counts, and patches `data.yaml:path` to Colab. NOTE val/test carry ~zero laptop/wallet/bottle - `best.pt`/early-stopping is phone/cup-driven; spot-check new classes manually.

**Colab setup:** `colab.research.google.com` -> Upload this notebook -> Runtime -> Change runtime type -> **T4 GPU** -> Runtime -> Run all (authorise Drive when asked). Upload `detectset_split.zip` to `Drive/Thesis_Workspace/datasets/` first (local copy: `src/datasets/detectset_split.zip`, ~1.2 GB).

**Session loop (free tier disconnects!):** run setup -> stage -> train (no seed cell here). On disconnect, re-run setup (mount only), then setup -> stage (unzip is skipped if present) -> resume instead of train. Always sync weights to Drive (train/resume do it automatically).

**Recipe:** Ultralytics defaults for augmentation (same `_COMMON` as the grafted notebook, ablated variable = head init only). (`mosaic=1.0`, `close_mosaic=10`, `scale=0.5`, `translate=0.1`, `fliplr=0.5`, `hsv 0.015/0.7/0.4`; `copy_paste` removed - proven no-op for box-only detect) + same two-phase schedule (frozen-backbone warmup, then full fine-tune). Deliberate non-defaults, all documented: `SGD + lr0=0.01 + cos_lr` (classic YOLO recipe), `patience=20` (Colab-budget early stopping vs default 100), `freeze=10` (backbone convention shared with `model_train_pose.ipynb`). Low-RAM T4 safe: `batch=16`, `workers=2`, `cache=disk` - record the observed per-epoch `GPU_mem` for the thesis.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q "ultralytics==8.4.127" nest_asyncio
!nvidia-smi -L
!free -h | head -3  # want 25GB+ total; ~13GB runtimes must use the disk-cache settings below

from pathlib import Path
import torch, ultralytics
import nest_asyncio
nest_asyncio.apply()  # lets ultralytics run inside Jupyter's event loop

# Self-defined run classes. IDs derive from order - never reorder, only append/replace deliberately.
# Canonical COCO naming (tagger rebound phone -> cell phone). wallet = manual-only (no COCO match).
SELECTED = ["cell phone", "laptop", "wallet", "bottle", "bag", "cup"]
BASE = "yolo11x.pt"  # x-arch COCO source (auto-downloads on first run)

PROJECT = Path("/content/drive/MyDrive/Thesis_Workspace")
DRIVE_DATASETS = PROJECT / "datasets"
SPLIT_ZIP_NAME = "detectset_split.zip"  # zipped src/datasets/detectset_split/ (images+labels+data.yaml)
SPLIT_ZIP_DRIVE = DRIVE_DATASETS / SPLIT_ZIP_NAME
SPLIT_LOCAL = Path("/content/datasets/detectset_split")
DATA_SRC = None  # set by the stage cell: unzipped data.yaml (train 3070 / val 170 / test 171)
RUNS_DIR = Path("/content/runs")
RUN_NAME = "standard_11X_colab"

assert len(set(SELECTED)) == len(SELECTED), f"SELECTED must be unique: {SELECTED}"
print(f"SELECTED ({len(SELECTED)}): {SELECTED}")
print("CLASS_MAP:", {name: i for i, name in enumerate(SELECTED)})

assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4."
print(torch.cuda.get_device_name(0))
print(f"ultralytics {ultralytics.__version__} | torch {torch.__version__}")
assert SPLIT_ZIP_DRIVE.exists(), \
    f"Upload {SPLIT_ZIP_NAME} to Drive/Thesis_Workspace/datasets/ first (local: src/datasets/detectset_split.zip)."
device = 0


In [ ]:
# Stage detectset_split.zip on Colab disk (unzip once, skip if present), validate counts +
# image<->label pairing, patch data.yaml path to Colab, verify names == SELECTED.
import shutil
import zipfile
import yaml as _yaml

Path("/content/datasets").mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

if not (SPLIT_LOCAL / "data.yaml").exists():
    print(f"Unzipping {SPLIT_ZIP_DRIVE} -> /content/datasets/ ...")
    with zipfile.ZipFile(SPLIT_ZIP_DRIVE, "r") as _z:
        _z.extractall("/content/datasets")
else:
    print(f"Split already present: {SPLIT_LOCAL} (delete it to force re-unzip)")

DATA_SRC = (SPLIT_LOCAL / "data.yaml").resolve()
assert DATA_SRC.is_file(), f"data.yaml missing after unzip: {DATA_SRC}"

# Validate counts (merged 2026-10-06: train 1475 vid* + 1595 Angle* = 3070; val 170; test 171).
_counts = {}
for _split in ["train", "val", "test"]:
    _imgs = sorted((SPLIT_LOCAL / "images" / _split).glob("*.jpg"))
    _lbls = sorted((SPLIT_LOCAL / "labels" / _split).glob("*.txt"))
    _counts[_split] = (len(_imgs), len(_lbls))
    _is = {p.stem for p in _imgs}
    _ls = {p.stem for p in _lbls}
    assert _is == _ls, f"stem mismatch in {_split}"
print(f"counts (images, labels): {_counts}")
assert _counts["train"] == (3070, 3070), f"unexpected train counts: {_counts['train']}"
assert _counts["val"] == (170, 170), f"unexpected val counts: {_counts['val']}"
assert _counts["test"] == (171, 171), f"unexpected test counts: {_counts['test']}"

# Validate + patch data.yaml (local absolute path C:/... -> Colab path; keep relative train/val/test).
_cfg = _yaml.safe_load(DATA_SRC.read_text(encoding="utf-8"))
print(f"data.yaml before: {_cfg}")
_names = _cfg.get("names")
_ordered = [_names[i] for i in sorted(_names)] if isinstance(_names, dict) else list(_names)
assert _ordered == SELECTED, f"data.yaml names {_ordered} != SELECTED {SELECTED}"
_cfg["path"] = SPLIT_LOCAL.as_posix()
_cfg.setdefault("train", "images/train")
_cfg.setdefault("val", "images/val")
_cfg.setdefault("test", "images/test")
DATA_SRC.write_text(_yaml.safe_dump(_cfg, sort_keys=False), encoding="utf-8")
print(f"data.yaml after: {_cfg}")
print(f"DATA_SRC -> {DATA_SRC}")


In [ ]:
# Two-phase train straight from BASE (overwrite baseline). Low-RAM T4 safe: batch=16, workers=2, cache on disk.
# Ultralytics resets the head nc=80 -> nc=6 here (random init); backbone/neck transfer.
# Phase 1: frozen-backbone warmup (fresh head settles, backbone intact).
# Phase 2: full fine-tune from phase-1 weights.
from ultralytics import YOLO
import shutil
from pathlib import Path as _P

FREEZE = 10          # backbone depth (same convention as model_train_pose FREEZE=10)
FREEZE_EPOCHS = 10
EPOCHS = 100

assert DATA_SRC and DATA_SRC.is_file(), "Run the stage cell first (DATA_SRC not set)."
_COMMON = dict(
    data=str(DATA_SRC),
    device=device,
    amp=True,
    imgsz=640,
    batch=16,
    workers=2,
    optimizer="SGD",
    lr0=0.01,
    cos_lr=True,
    mosaic=1.0,
    close_mosaic=10,
    scale=0.5,
    translate=0.1,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    cache="disk",
    plots=True,
    save_period=10,
    project=str(RUNS_DIR),
    patience=20,
    verbose=True,
    seed=0,
)

_m1 = YOLO(BASE)  # auto-downloads; nc-override happens at train start
_m1.train(epochs=FREEZE_EPOCHS, freeze=FREEZE, name=RUN_NAME, exist_ok=False, **_COMMON)
_p1_last = _P(_m1.trainer.save_dir) / "weights" / "last.pt"
assert _p1_last.exists(), f"phase-1 last.pt missing: {_p1_last}"
print(f"Phase-1 (frozen warmup) done -> {_p1_last}")

_m2 = YOLO(str(_p1_last))
_m2.train(epochs=EPOCHS, name=RUN_NAME + "-full", exist_ok=False, **_COMMON)

# Colab disks are ephemeral: sync EVERY matching run dir (warmup + full + bumps).
_synced = []
for _run in sorted(RUNS_DIR.glob(RUN_NAME + "*")):
    if not (_run.is_dir() and _run.parent == RUNS_DIR and (_run / "weights").is_dir()):
        continue
    _dst = PROJECT / "runs" / "detect" / _run.name
    _dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(_run, _dst, dirs_exist_ok=True)
    _synced.append(_run.name)
    _last = _dst / "weights" / "last.pt"
    assert _last.exists(), f"last.pt missing after sync: {_dst}"
    _best = _dst / "weights" / "best.pt"
    _msg = (f"best.pt {_best.stat().st_size/1e6:.1f} MB"
            if _best.exists() and _best.stat().st_size > 40_000_000
            else "last.pt only (partial run — resume it, don't finish)")
    print(f"Synced {_run.name} -> {_dst} | {_msg}")
assert _synced, "No run dirs synced — training produced nothing?"


In [ ]:
!ls -la /content/runs/ 2>/dev/null; du -sh /content/runs/standard_11X_colab*/weights/ 2>/dev/null

In [ ]:
# Manual weights copy: newest standard_* checkpoint -> Drive (auto-finds the run dir).
import shutil
from pathlib import Path
_cands = sorted(
    (p for p in Path("/content/runs").glob("standard_11X_colab*")
     if (p / "weights" / "last.pt").exists()),
    key=lambda p: (p / "weights" / "last.pt").stat().st_mtime, reverse=True)
assert _cands, "No checkpointed selected_* run under /content/runs."
src = _cands[0] / "weights"
dst = Path("/content/drive/MyDrive/Thesis_Workspace/runs/detect") / _cands[0].name / "weights"
dst.mkdir(parents=True, exist_ok=True)
for f in ["best.pt", "last.pt"]:
    if not (src / f).exists():
        print(f"{f} not yet available, skipping")
        continue
    shutil.copy(src / f, dst / f)
    print(f, f"{(dst / f).stat().st_size/1e6:.1f} MB on Drive (from {_cands[0].name})")


In [ ]:
# Resume after a disconnect: run the setup cell (mount) + THIS cell, not the train cell.
# Finds the newest local checkpoint, else pulls the newest one from Drive.
from ultralytics import YOLO
import shutil
from pathlib import Path as _P

RUNS_MIRROR = PROJECT / "runs" / "detect"
_locals = sorted(
    (p for p in RUNS_DIR.glob("standard_11X_colab*")
     if (p / "weights" / "last.pt").exists()),
    key=lambda p: (p / "weights" / "last.pt").stat().st_mtime, reverse=True)
if _locals:
    ckpt = _locals[0] / "weights" / "last.pt"
    print(f"Resuming local: {ckpt}")
else:
    _remote = sorted(
        (p for p in RUNS_MIRROR.glob("standard_11X_colab*")
         if (p / "weights" / "last.pt").exists()),
        key=lambda p: (p / "weights" / "last.pt").stat().st_mtime,
        reverse=True) if RUNS_MIRROR.exists() else []
    assert _remote, "No checkpoint locally or on Drive — run the train cell first."
    ckpt = RUNS_DIR / _remote[0].name / "weights" / "last.pt"
    ckpt.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(_remote[0] / "weights" / "last.pt", ckpt)
    print(f"Pulled checkpoint from Drive: {_remote[0]}")

model = YOLO(str(ckpt))
model.train(resume=True)

# Sync back the resumed run dir.
_run_dir = _P(getattr(getattr(model, "trainer", None), "save_dir", "") or "")
if not _run_dir or not _run_dir.exists():
    _cands = sorted((p for p in RUNS_DIR.glob("standard_11X_colab*") if p.is_dir()),
                    key=lambda p: p.stat().st_mtime, reverse=True)
    assert _cands, "No run dir found under /content/runs!"
    _run_dir = _cands[0]
assert _run_dir.is_dir() and _run_dir.parent == RUNS_DIR and (_run_dir / "weights").is_dir(), \
    f"Refusing to sync unexpected dir: {_run_dir}"
print(f"Actual run dir: {_run_dir}")

_dst = PROJECT / "runs" / "detect" / _run_dir.name
_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(_run_dir, _dst, dirs_exist_ok=True)
_best = _dst / "weights" / "best.pt"
assert _best.exists() and _best.stat().st_size > 40_000_000, \
    f"best.pt missing/too small after sync: {_best} (sync copied the wrong dir?)"
print(f"Synced {_run_dir.name} -> {_dst} | best.pt {_best.stat().st_size/1e6:.1f} MB")


In [ ]:
# Finish: publish best.pt to Drive + honest val numbers. Auto-finds the newest
# complete run locally or on Drive, so renamed/bumped run dirs just work.
from ultralytics import YOLO
import shutil
from pathlib import Path as _P

_roots = [d for d in (RUNS_DIR, PROJECT / "runs" / "detect") if d.exists()]
_cands = sorted(
    (p for d in _roots for p in d.glob("standard_11X_colab*")
     if (p / "weights" / "best.pt").exists()
     and (p / "weights" / "best.pt").stat().st_size > 40_000_000),
    key=lambda p: (p / "weights" / "best.pt").stat().st_mtime, reverse=True)
assert _cands, "No complete best.pt anywhere — train/resume first."
best = _cands[0] / "weights" / "best.pt"
out = PROJECT / "models" / "detect" / "best_standard_colab.pt"
out.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(best, out)
assert out.stat().st_size > 40_000_000
print(f"Published {out} ({out.stat().st_size/1e6:.1f} MB) from {_cands[0].name}")

m = YOLO(str(best))
print(f"names: {m.names}")
print(m.val(data=str(DATA_SRC), device=device))  # full val split (plots=True run already saved confusion matrix + PR curves)
import yaml as _yaml
_cfg = _yaml.safe_load(DATA_SRC.read_text(encoding="utf-8"))
if _cfg.get("test"):
    print(m.val(data=str(DATA_SRC), device=device, split="test"))  # held-out test, only if the ndjson carried a test split
else:
    print("No test split in DATA_SRC — val above is the reported metric (add test records to the ndjson for held-out evaluation).")


In [ ]:
# ONE-CLICK RESCUE: copy the newest CHECKPOINTED run to Drive.
# Run this if a finished (or crashed) training never synced. Partial runs
# (last.pt, no best.pt yet) are rescued too.
import shutil
from pathlib import Path as _P

_found = sorted(
    (p for p in RUNS_DIR.glob("standard_11X_colab*")
     if (p / "weights" / "last.pt").exists()),
    key=lambda p: (p / "weights" / "last.pt").stat().st_mtime, reverse=True)
assert _found, "No checkpointed run (weights/last.pt) under /content/runs."
_src = _found[0]
_dst = PROJECT / "runs" / "detect" / _src.name
shutil.copytree(_src, _dst, dirs_exist_ok=True)
_last = _dst / "weights" / "last.pt"
assert _last.exists(), "rescue copy failed!"
_best = _dst / "weights" / "best.pt"
_msg = (f"best.pt {_best.stat().st_size/1e6:.1f} MB"
        if _best.exists() and _best.stat().st_size > 40_000_000
        else "no complete best.pt yet (partial run)")
print(f"Rescued {_src.name} -> {_dst} | last.pt OK | {_msg}")
